In [ ]:
# 1. IMPORT LIBRARY
import os, re, sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import fitz  # PyMuPDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score)
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Lazy init EasyOCR (hanya kalau PyMuPDF gagal dapat teks)
reader = None

def get_ocr_reader():
    """Inisialisasi EasyOCR secara lazy untuk hemat waktu & memori."""
    global reader
    if reader is None:
        try:
            import easyocr
            print('Loading EasyOCR model (first time)...')
            reader = easyocr.Reader(['id', 'en'], gpu=True)
        except ImportError:
            print('[WARNING] easyocr tidak terinstall. Scanned PDF tidak bisa diproses.')
            reader = False
    return reader

print('Library berhasil diimport!')


## 2. Baca PDF dari Folder Dataset


In [ ]:
# 2. Baca PDF dari Folder Dataset

# Path ke folder dataset (coba beberapa lokasi yang umum)
possible_dataset_paths = [
    Path('./dataset'),
    Path('../dataset'),
    Path(__file__).parent / 'dataset' if '__file__' in globals() else Path('./dataset'),
]

DATASET_DIR = None
for p in possible_dataset_paths:
    if p.exists() and p.is_dir():
        DATASET_DIR = p
        break

if DATASET_DIR is None:
    print('WARNING: folder dataset tidak ditemukan!')
    print(f'CWD saat ini: {os.getcwd()}')
    print(f'Isi folder: {os.listdir(".")}')
    DATASET_DIR = Path('./dataset')

print(f'Menggunakan dataset dari: {DATASET_DIR.resolve()}')

# Mapping jenis dokumen ke arah (berdasarkan SOP PT. Almex Bintang Timur)
JENIS_KE_ARAH = {
    'PurchaseOrder': 'Masuk',
    'Invoice': 'Keluar',
    'Penawaran': 'Keluar',
    'Sales Order': 'Keluar',
    'SuratJalan': 'Keluar',
}

PDF_EXT = {'.pdf'}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}
SUPPORTED_EXT = PDF_EXT | IMG_EXT

def ocr_image(img_np):
    """OCR satu gambar (numpy array) pakai EasyOCR (lazy load)."""
    rdr = get_ocr_reader()
    if not rdr:
        return ''
    try:
        results = rdr.readtext(img_np, detail=0)
        return ' '.join(results)
    except Exception as e:
        print(f'    [OCR ERROR] {e}')
        return ''

def extract_text_from_file(file_path):
    """
    Ekstrak teks dari PDF atau IMAGE.
    - PDF text-based: langsung ambil teks via PyMuPDF
    - PDF scanned (image): render halaman jadi gambar, lalu OCR
    - IMAGE (jpg/png/etc): langsung OCR
    """
    ext = os.path.splitext(file_path)[1].lower()
    text = ''

    try:
        if ext in PDF_EXT:
            doc = fitz.open(file_path)
            for page in doc:
                page_text = page.get_text().strip()
                if page_text:
                    text += page_text + '\n'
                else:
                    # Scanned PDF: render ke gambar, lalu OCR
                    import cv2
                    pix = page.get_pixmap(dpi=200)  # turunkan DPI biar cepat
                    img_bytes = pix.tobytes('png')
                    img_np = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_COLOR)
                    if img_np is not None:
                        ocr_text = ocr_image(img_np)
                        if ocr_text:
                            text += ocr_text + '\n'
            doc.close()

        elif ext in IMG_EXT:
            import cv2
            img_np = cv2.imread(file_path)
            if img_np is not None:
                text += ocr_image(img_np) + '\n'
            else:
                print(f'  [ERROR] Gagal baca gambar: {os.path.basename(file_path)}')

    except Exception as e:
        print(f'  [ERROR] {os.path.basename(file_path)}: {e}')

    return text.strip()

# Baca semua file (PDF + IMAGE)
data = []
print('Membaca file dari folder dataset...')
print('=' * 60)

for folder_name in sorted(os.listdir(DATASET_DIR)):
    folder_path = DATASET_DIR / folder_name
    if not folder_path.is_dir():
        continue

    jenis = folder_name
    # Arah ditentukan dari jenis folder
    arah = JENIS_KE_ARAH.get(jenis, 'Keluar')

    all_files = [f for f in os.listdir(folder_path)
                 if Path(f).suffix.lower() in SUPPORTED_EXT]
    
    pdf_count = sum(1 for f in all_files if Path(f).suffix.lower() in PDF_EXT)
    img_count = sum(1 for f in all_files if Path(f).suffix.lower() in IMG_EXT)
    print(f'\nFolder: {folder_name} ({len(all_files)} file: {pdf_count} PDF, {img_count} gambar)')

    for file_name in sorted(all_files):
        file_path = folder_path / file_name
        text = extract_text_from_file(str(file_path))

        if not text:
            print(f'  [SKIP] {file_name} - teks kosong')
            continue

        data.append({
            'file': file_name,
            'text': text,
            'arah': arah,
            'jenis': jenis,
        })
        file_type = 'PDF' if Path(file_name).suffix.lower() in PDF_EXT else 'IMG'
        print(f'  [OK] {file_name} [{file_type}] -> {jenis} ({arah})')

df = pd.DataFrame(data)
print(f'\n{"=" * 60}')
print(f'Total file terbaca: {len(df)}')
print(f'\nDistribusi Jenis:')
print(df['jenis'].value_counts().to_string())
print(f'\nDistribusi Arah:')
print(df['arah'].value_counts().to_string())


## 3. Cek Data

Pastikan setiap kategori punya minimal 5 dokumen. Kalau kurang, tambahkan PDF.


In [ ]:
# 3. Validasi Data
print('Validasi dataset:')
print('=' * 60)
for jenis, count in df['jenis'].value_counts().items():
    status = 'OK' if count >= 5 else 'KURANG (min 5)'
    print(f'  {jenis:25s} : {count:3d} dokumen  [{status}]')

print(f'\nArah:')
for arah, count in df['arah'].value_counts().items():
    print(f'  {arah:25s} : {count:3d} dokumen')

if len(df) < 30:
    print(f'\n[WARNING] Total data terlalu sedikit ({len(df)}). Disarankan minimal 50 dokumen.')
else:
    print(f'\n[OK] Total data cukup: {len(df)} dokumen')


## 4. Preprocessing Teks


In [ ]:
# 4. Preprocessing Teks
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

stopword_factory = StopWordRemoverFactory()
stopword_list = stopword_factory.get_stop_words()

# Gabungkan stopword Sastrawi + custom
# Hapus semua angka 4 digit (tahun) di preprocessing, bukan hardcode tahun
company_stopwords = {
    'pt', 'cv', 'tbk', 'abt', 'vi',
    'nomor', 'perihal', 'lampiran', 'kepada', 'yth',
}

# Kata-kata yang muncul di hampir semua dokumen / tidak membantu bedain jenis
# (hanya membuat TF-IDF top feature jadi generic)
domain_stopwords = {
    'rucika', 'pcs', 'batang', 'total', 'harga', 'diskon', 'tanggal', 'kode',
    'barang', 'nama', 'qty', 'satuan', 'rupiah', 'ratus', 'ribu', 'juta',
    'belas', 'puluh', 'indonesia', 'tangerang', 'banten', 'kota', 'green',
    'lake', 'city', 'ruko', 'timur', 'bintang', 'almex', 'jan', 'feb', 'mar',
    'apr', 'mei', 'jun', 'jul', 'agu', 'sep', 'okt', 'nov', 'des', 'no',
    'jumlah', 'sub', 'lain', 'biaya', 'ppn', 'dpp', 'net', 'cash', 'transfer',
    'dibuat', 'disetujui', 'pengirim', 'penerima', 'keterangan',
}

custom_stopwords = set(stopword_list) | company_stopwords | domain_stopwords

def preprocess_text(text):
    text = text.lower()
    # Hapus angka (termasuk tahun), tanda baca, simbol
    text = re.sub(r'\b\d{4}\b', ' ', text)  # hapus tahun 4 digit
    text = re.sub(r'[^a-z\s]', ' ', text)    # hapus non-huruf
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    # Filter stopword & panjang minimum
    tokens = [t for t in tokens if t not in custom_stopwords and len(t) > 2]
    # Stemming bahasa Indonesia
    if tokens:
        tokens = stemmer.stem(' '.join(tokens)).split()
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess_text)

print('Contoh hasil preprocessing:')
print('=' * 60)
for i in range(min(3, len(df))):
    print(f'\nFile    : {df.iloc[i]["file"]}')
    print(f'Original: {df.iloc[i]["text"][:120]}...')
    print(f'Cleaned : {df.iloc[i]["clean_text"][:120]}...')
    print(f'Jenis   : {df.iloc[i]["jenis"]}  |  Arah: {df.iloc[i]["arah"]}')


## 5. Ekstraksi Fitur TF-IDF


In [ ]:
# 5. Split Data & Ekstraksi Fitur TF-IDFn_docs = len(df)# Lebih agresif buang kata yang terlalu umum, supaya top feature tidak genericmax_features = min(800, max(300, n_docs * 8))min_df = 3 if n_docs >= 80 else 2max_df = 0.70  # buang kata yang muncul di >70% dokumen# Split DULU sebelum TF-IDFX_text = df['clean_text'].tolist()y_arah = df['arah']y_jenis = df['jenis']X_train_text, X_test_text, y_train_arah, y_test_arah, y_train_jenis, y_test_jenis = train_test_split(    X_text, y_arah, y_jenis, test_size=0.2, random_state=42, stratify=y_jenis)print(f'Total data     : {len(X_text)} dokumen')print(f'Training (80%) : {len(X_train_text)} dokumen')print(f'Testing (20%)  : {len(X_test_text)} dokumen')print()print('Distribusi Arah (Training):')print(y_train_arah.value_counts().to_string())print()print('Distribusi Arah (Testing):')print(y_test_arah.value_counts().to_string())print()print('Distribusi Jenis (Training):')print(y_train_jenis.value_counts().to_string())# TF-IDF fit hanya pada training settfidf_vectorizer = TfidfVectorizer(    max_features=max_features,    ngram_range=(1, 2),    min_df=min_df,    max_df=max_df,    sublinear_tf=True,    use_idf=True)X_train = tfidf_vectorizer.fit_transform(X_train_text)X_test = tfidf_vectorizer.transform(X_test_text)print(f'\nShape matriks TF-IDF (train): {X_train.shape}')print(f'Shape matriks TF-IDF (test) : {X_test.shape}')print(f'Jumlah fitur: {len(tfidf_vectorizer.get_feature_names_out())}')print(f'min_df: {min_df}, max_df: {max_df}, max_features: {max_features}')print()feature_names = tfidf_vectorizer.get_feature_names_out()# (Opsional) Top fitur berdasarkan total bobot — hanya menunjukkan kata paling seringtfidf_sum = X_train.sum(axis=0).A1top_indices = tfidf_sum.argsort()[-20:][::-1]print('Top 20 fitur TF-IDF (by total weight, kurang informatif):')for idx in top_indices:    print(f'  {feature_names[idx]:25s} -> bobot: {tfidf_sum[idx]:.4f}')print()# Lebih informatif: fitur paling membedakan tiap jenis (chi2)from sklearn.feature_selection import chi2print('Top 10 fitur paling membedakan per jenis (chi2):')print('=' * 60)for jenis in sorted(y_train_jenis.unique()):    y_binary = (y_train_jenis == jenis).astype(int)    chi2_scores, p_values = chi2(X_train, y_binary)    top_idx = chi2_scores.argsort()[-10:][::-1]    print(f'\n{jenis}:')    for idx in top_idx:        print(f'  {feature_names[idx]:25s} -> chi2: {chi2_scores[idx]:.2f}')# Matriks X inilah yang jadi input Naive Bayesprint('\nMatriks TF-IDF siap dipakai Naive Bayes untuk arah & jenis.')

## 6. Split Data (80% Training, 20% Testing)


In [ ]:
# 6. Split Data (80% Training, 20% Testing)
min_samples_jenis = y_jenis.value_counts().min()
min_samples_arah = y_arah.value_counts().min()

# Stratified split hanya jika ada cukup data per kelas
test_size = 0.2
if min(min_samples_jenis, min_samples_arah) < 5:
    test_size = 0.1
    print('[INFO] Data sedikit, test_size diturunkan jadi 10%')

X_train_arah, X_test_arah, y_train_arah, y_test_arah = train_test_split(
    X, y_arah, test_size=test_size, random_state=42, stratify=y_arah
)

X_train_jenis, X_test_jenis, y_train_jenis, y_test_jenis = train_test_split(
    X, y_jenis, test_size=test_size, random_state=42, stratify=y_jenis
)

print(f'Total data     : {X.shape[0]} dokumen')
print(f'Training ({int((1-test_size)*100)}%) : {X_train_arah.shape[0]} dokumen')
print(f'Testing ({int(test_size*100)}%)  : {X_test_arah.shape[0]} dokumen')
print()
print('Distribusi Arah (Training):')
print(y_train_arah.value_counts().to_string())
print()
print('Distribusi Arah (Testing):')
print(y_test_arah.value_counts().to_string())
print()
print('Distribusi Jenis (Training):')
print(y_train_jenis.value_counts().to_string())


## 7. Training Model - Klasifikasi Arah Dokumen


In [ ]:
# 7. Training Model - Klasifikasi Arah Dokumen

# Hitung class prior untuk handle imbalance (20 Masuk vs 80 Keluar)
arah_counts = y_train_arah.value_counts()
classes = sorted(arah_counts.index)
prior = np.array([arah_counts[c] for c in classes], dtype=float)
prior = prior / prior.sum()

print('Class prior (arah):')
for c, p in zip(classes, prior):
    print(f'  {c}: {p:.4f}')
print()

model_arah = MultinomialNB(alpha=1.0, fit_prior=True)
model_arah.fit(X_train, y_train_arah)

y_pred_arah = model_arah.predict(X_test)

print('KLASIFIKASI ARAH DOKUMEN (Masuk/Keluar)')
print('=' * 50)
print(f'Accuracy : {accuracy_score(y_test_arah, y_pred_arah):.4f}')
print(f'Precision: {precision_score(y_test_arah, y_pred_arah, average="weighted"):.4f}')
print(f'Recall   : {recall_score(y_test_arah, y_pred_arah, average="weighted"):.4f}')
print(f'F1-Score : {f1_score(y_test_arah, y_pred_arah, average="weighted"):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test_arah, y_pred_arah))


## 8. Confusion Matrix - Arah Dokumen


In [ ]:
cm_arah = confusion_matrix(y_test_arah, y_pred_arah, labels=model_arah.classes_)



plt.figure(figsize=(8, 6))

sns.heatmap(cm_arah, annot=True, fmt='d', cmap='Blues',

            xticklabels=model_arah.classes_, yticklabels=model_arah.classes_,

            linewidths=0.5, linecolor='gray')

plt.title('Confusion Matrix - Klasifikasi Arah Dokumen', fontsize=14, fontweight='bold')

plt.xlabel('Prediksi', fontsize=12)

plt.ylabel('Aktual', fontsize=12)

plt.tight_layout()

plt.savefig('confusion_matrix_arah.png', dpi=150, bbox_inches='tight')

plt.show()

print('Gambar disimpan: confusion_matrix_arah.png')


## 9. Training Model - Klasifikasi Jenis Dokumen


In [ ]:
# 9. Training Model - Klasifikasi Jenis Dokumen
model_jenis = MultinomialNB(alpha=1.0)
model_jenis.fit(X_train, y_train_jenis)

y_pred_jenis = model_jenis.predict(X_test)

print('KLASIFIKASI JENIS DOKUMEN')
print('=' * 50)
print(f'Accuracy : {accuracy_score(y_test_jenis, y_pred_jenis):.4f}')
print(f'Precision: {precision_score(y_test_jenis, y_pred_jenis, average="weighted"):.4f}')
print(f'Recall   : {recall_score(y_test_jenis, y_pred_jenis, average="weighted"):.4f}')
print(f'F1-Score : {f1_score(y_test_jenis, y_pred_jenis, average="weighted"):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test_jenis, y_pred_jenis))


## 10. Confusion Matrix - Jenis Dokumen


In [ ]:
cm_jenis = confusion_matrix(y_test_jenis, y_pred_jenis, labels=model_jenis.classes_)



plt.figure(figsize=(10, 8))

sns.heatmap(cm_jenis, annot=True, fmt='d', cmap='Oranges',

            xticklabels=model_jenis.classes_, yticklabels=model_jenis.classes_,

            linewidths=0.5, linecolor='gray')

plt.title('Confusion Matrix - Klasifikasi Jenis Dokumen', fontsize=14, fontweight='bold')

plt.xlabel('Prediksi', fontsize=12)

plt.ylabel('Aktual', fontsize=12)

plt.xticks(rotation=45, ha='right')

plt.yticks(rotation=0)

plt.tight_layout()

plt.savefig('confusion_matrix_jenis.png', dpi=150, bbox_inches='tight')

plt.show()

print('Gambar disimpan: confusion_matrix_jenis.png')


## 11. Cross-Validation (5-Fold)


In [ ]:
# 11. Cross-Validation (Stratified K-Fold)min_class_size = min(y_jenis.value_counts().min(), y_arah.value_counts().min())n_folds = min(5, max(2, min_class_size))print(f'Menggunakan StratifiedKFold dengan {n_folds} folds')print()# Gunakan Pipeline agar TF-IDF fit hanya di training fold setiap iterasi (no leakage)from sklearn.pipeline import Pipelinecv_arah = cross_val_score(    Pipeline([        ('tfidf', TfidfVectorizer(max_features=max_features, ngram_range=(1, 2),                                   min_df=min_df, max_df=max_df, sublinear_tf=True)),        ('clf', MultinomialNB(alpha=1.0))    ]),    df['clean_text'], y_arah,    cv=StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42),    scoring='accuracy')cv_jenis = cross_val_score(    Pipeline([        ('tfidf', TfidfVectorizer(max_features=max_features, ngram_range=(1, 2),                                   min_df=min_df, max_df=max_df, sublinear_tf=True)),        ('clf', MultinomialNB(alpha=1.0))    ]),    df['clean_text'], y_jenis,    cv=StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42),    scoring='accuracy')print(f'Hasil {n_folds}-Fold Cross-Validation:')print('=' * 50)print(f'\nArah Dokumen:')for i, score in enumerate(cv_arah, 1):    print(f'  Fold {i}: {score:.4f}')print(f'  Mean : {cv_arah.mean():.4f}')print(f'  Std  : {cv_arah.std():.4f}')print(f'\nJenis Dokumen:')for i, score in enumerate(cv_jenis, 1):    print(f'  Fold {i}: {score:.4f}')print(f'  Mean : {cv_jenis.mean():.4f}')print(f'  Std  : {cv_jenis.std():.4f}')fig, axes = plt.subplots(1, 2, figsize=(12, 5))axes[0].bar(range(1, n_folds+1), cv_arah, color='#3B82F6', alpha=0.8)axes[0].axhline(y=cv_arah.mean(), color='red', linestyle='--', label=f'Mean: {cv_arah.mean():.4f}')axes[0].set_title('Cross-Validation - Arah Dokumen', fontweight='bold')axes[0].set_xlabel('Fold'); axes[0].set_ylabel('Accuracy'); axes[0].set_ylim(0, 1.1); axes[0].legend()axes[1].bar(range(1, n_folds+1), cv_jenis, color='#F59E0B', alpha=0.8)axes[1].axhline(y=cv_jenis.mean(), color='red', linestyle='--', label=f'Mean: {cv_jenis.mean():.4f}')axes[1].set_title('Cross-Validation - Jenis Dokumen', fontweight='bold')axes[1].set_xlabel('Fold'); axes[1].set_ylabel('Accuracy'); axes[1].set_ylim(0, 1.1); axes[1].legend()plt.tight_layout()plt.savefig('cross_validation.png', dpi=150, bbox_inches='tight')plt.show()print('Gambar disimpan: cross_validation.png')

## 12. Simpan Model (untuk backend)


In [ ]:
# 12. Simpan Model (untuk backend)
import joblib

# Deteksi lokasi repo secara otomatis
current_dir = Path(os.getcwd())
# Coba cari backend/ml_model dari lokasi notebook
possible_model_dirs = [
    current_dir / '..' / 'backend' / 'ml_model',
    current_dir / 'backend' / 'ml_model',
    Path('/root/pengarsipan-almex-bintang-timur/backend/ml_model'),
]

MODEL_DIR = None
for p in possible_model_dirs:
    if (p.parent / 'main.py').exists() or (p.parent / 'classifier.py').exists() or p.parent.name == 'backend':
        MODEL_DIR = p
        break

if MODEL_DIR is None:
    MODEL_DIR = current_dir / 'ml_model'

MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(model_arah, MODEL_DIR / 'arah_pipeline.pkl')
joblib.dump(model_jenis, MODEL_DIR / 'jenis_pipeline.pkl')
joblib.dump(tfidf_vectorizer, MODEL_DIR / 'tfidf_vectorizer.pkl')

print('Model berhasil disimpan!')
print(f'  -> {MODEL_DIR / "arah_pipeline.pkl"}')
print(f'  -> {MODEL_DIR / "jenis_pipeline.pkl"}')
print(f'  -> {MODEL_DIR / "tfidf_vectorizer.pkl"}')
print(f'\nModel siap dipakai di backend. Restart server untuk memuat model baru.')
